In [2]:
import duckdb
import glob

Checking if everything works

In [2]:
df = duckdb.query("""
    SELECT *
    FROM read_csv_auto('C:\\Users\\bramm\\studen-grade data\\Data\\raw\\EdNet-KT1\KT1\\*.csv')    
    LIMIT 100
""").to_df()

print(df.head(10))
print(df.columns)

<>:1: SyntaxWarning: invalid escape sequence '\K'
<>:1: SyntaxWarning: invalid escape sequence '\K'
C:\Users\bramm\AppData\Local\Temp\ipykernel_29744\2977817159.py:1: SyntaxWarning: invalid escape sequence '\K'
  df = duckdb.query("""


IOException: IO Error: No files found that match the pattern "C:\Users\bramm\studen-grade data\Data\raw\EdNet-KT1\KT1\*.csv"

Use duckdb to change the dataset into a parquet file for easier reading since the files are too clumsy for reading using python and csv.

In [ ]:

con = duckdb.connect()

con.execute("""
    COPY (
        SELECT 
            timestamp::BIGINT,
            solving_id::INT,
            question_id,
            user_answer,
            elapsed_time::INT
        FROM read_csv_auto('C:\\Users\\bramm\\studen-grade data\\Data\\raw\\EdNet-KT1\KT1\\*.csv')
        WHERE solving_id % 10 = 0
    )
    TO 'ednet_subset.parquet'
    (FORMAT CSV, COMPRESSION ZSTD);
""")


<>:3: SyntaxWarning: invalid escape sequence '\K'
<>:3: SyntaxWarning: invalid escape sequence '\K'
C:\Users\bramm\AppData\Local\Temp\ipykernel_35308\803476173.py:3: SyntaxWarning: invalid escape sequence '\K'
  con.execute("""


RuntimeError: Query interrupted

#### Combine files from kt1 into a single csv
I am using glob and duckdb to efficiently combine the files in the kt1 directory into a single csv, loading many small files is much less efficient than loading 1 bigger csv.

In [4]:
files = glob.glob('C:\\Users\\bramm\\S3C2-Data\\Student grade project\\Data\\raw\\EdNet-KT1\\KT1\\*.csv')
print(f"Found {len(files)} files.")

# Select the first x amount of files for processing
subset_files = files[:5000]

con = duckdb.connect()

file_list = ','.join([f"'{file}'" for file in subset_files])

con.execute(f"""
    COPY (
        SELECT 
            CAST(timestamp AS BIGINT),
            CAST(solving_id AS INT),
            question_id,
            user_answer,
            CAST(elapsed_time AS INT)
        FROM read_csv([{file_list}])
        WHERE solving_id % 10 = 0
    )
    TO 'ednet_kt1_subset.csv'
    (FORMAT CSV);
""")

Found 784309 files.


In [6]:
files = glob.glob('C:\\Users\\bramm\\S3C2-Data\\Student grade project\\Data\\raw\\EdNet-KT2\\KT2\\*.csv')
print(f"Found {len(files)} files.")

# Select the first x amount of files for processing
subset_files = files[:5000]

con = duckdb.connect()

file_list = ','.join([f"'{file}'" for file in subset_files])

con.execute(f"""
    COPY (
        SELECT 
            CAST(timestamp AS BIGINT),
            action_type,
            item_id,
            source,
            user_answer,
            platform
        FROM read_csv([{file_list}])
    )
    TO 'ednet_kt2_subset.csv'
    (FORMAT CSV);
""")

Found 297444 files.


In [7]:
files = glob.glob('C:\\Users\\bramm\\S3C2-Data\\Student grade project\\Data\\raw\\EdNet-KT3\\KT3\\*.csv')
print(f"Found {len(files)} files.")

# Select the first x amount of files for processing
subset_files = files[:5000]

con = duckdb.connect()

file_list = ','.join([f"'{file}'" for file in subset_files])

con.execute(f"""
    COPY (
        SELECT 
            CAST(timestamp AS BIGINT),
            action_type,
            item_id,
            source,
            user_answer,
            platform
        FROM read_csv([{file_list}])
    )
    TO 'ednet_kt3_subset.csv'
    (FORMAT CSV);
""")

Found 297915 files.
